In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ['OPENAI_BASE_URL'],
    api_key=os.environ['OPENAI_API_KEY'], # ModelScope Token
)

response = client.chat.completions.create(
    model=os.environ['OPENAI_MODEL'], # ModelScope Model-Id, required
    messages=[
        {
            'role': 'user',
            'content': '你好'
        }
    ],
    stream=True
)
done_reasoning = False
for chunk in response:
    if chunk.choices:
        reasoning_chunk = chunk.choices[0].delta.reasoning_content
        answer_chunk = chunk.choices[0].delta.content
        if reasoning_chunk != '':
            print(reasoning_chunk, end='', flush=True)
        elif answer_chunk != '':
            if not done_reasoning:
                print('\n\n === Final Answer ===\n')
                done_reasoning = True
            print(answer_chunk, end='', flush=True)

我们需要回答用户。用户说“你好”。需要简单友好回应。可能询问用户需要什么帮助。用中文。不需要复杂。

 === Final Answer ===

你好！😊 有什么可以帮你的吗？

In [2]:
import os
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

embedding_model = OpenAIEmbeddings(
    model=os.environ['EMBEDDING_MODEL'],
    base_url=os.environ['EMBEDDING_BASE_URL'],
    api_key=os.environ['EMBEDDING_API_KEY'],
    check_embedding_ctx_length=False,
)

embedding_model.embed_query("你好")

[0.0172119140625,
 -0.00928497314453125,
 0.005107879638671875,
 -0.03985595703125,
 -0.0021800994873046875,
 -0.01259613037109375,
 -0.0157470703125,
 0.021484375,
 -0.03155517578125,
 0.03570556640625,
 -0.0280609130859375,
 -0.0160369873046875,
 0.03924560546875,
 -0.0247955322265625,
 -0.0087127685546875,
 0.005901336669921875,
 -0.040740966796875,
 -0.0170440673828125,
 0.0286102294921875,
 0.0141143798828125,
 0.00884246826171875,
 -0.00930023193359375,
 0.00591278076171875,
 -0.01230621337890625,
 0.0235137939453125,
 0.0249176025390625,
 -0.031951904296875,
 -0.017791748046875,
 -0.01509857177734375,
 0.017974853515625,
 -0.0343017578125,
 -0.0027065277099609375,
 0.01338958740234375,
 0.0293121337890625,
 -0.0087890625,
 0.0305938720703125,
 -0.004039764404296875,
 -0.01349639892578125,
 0.01617431640625,
 -0.0228118896484375,
 0.0074615478515625,
 -0.00940704345703125,
 -0.00469970703125,
 -0.0155029296875,
 -0.00269317626953125,
 -0.0210418701171875,
 -0.027740478515625,
 -0

In [2]:
from langchain_core.documents import Document

# 1. 准备示例文档数据
docs = [
    Document(page_content="iPhone 15 Pro 采用了钛金属边框，搭载 A17 Pro 芯片。"),
    Document(page_content="苹果公司的最新智能手机电池续航得到了大幅提升。"),
    Document(page_content="华为 Mate 60 Pro 支持卫星通话功能，采用麒麟芯片。"),
    Document(page_content="特斯拉 Model 3 是一部纯电动轿车，续航里程较长。"),
]

In [3]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(docs, embedding_model)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [4]:
vector_retriever.invoke("卫星通话") 

[Document(id='7aad40b7-280f-4253-9043-a68629c31217', metadata={}, page_content='华为 Mate 60 Pro 支持卫星通话功能，采用麒麟芯片。'),
 Document(id='dbb08674-0b19-4cd8-88ba-c87f4e3f4e7a', metadata={}, page_content='苹果公司的最新智能手机电池续航得到了大幅提升。'),
 Document(id='240991ee-0c06-4146-b06e-3cce1b34177a', metadata={}, page_content='iPhone 15 Pro 采用了钛金属边框，搭载 A17 Pro 芯片。'),
 Document(id='a454288b-2a13-49cf-9b20-7695e025e5ad', metadata={}, page_content='特斯拉 Model 3 是一部纯电动轿车，续航里程较长。')]

In [6]:
import jieba
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers import EnsembleRetriever

# 2. 构建 BM25 检索器（稀疏检索）
# 注意：若针对中文，建议传入分词函数 preprocess_func=jieba.lcut
bm25_retriever = BM25Retriever.from_documents(docs, preprocess_func=jieba.lcut)
bm25_retriever.k = 5  # 单独召回前 5 条

# 4. 使用 EnsembleRetriever 进行 RRF 融合
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],  # 设置融合权重（默认为平等 RRF 融合）
)

# 5. 执行混合检索
query = "苹果手机的芯片和电池怎么样？"
results = ensemble_retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.page_content}")

[1] 苹果公司的最新智能手机电池续航得到了大幅提升。
[2] 特斯拉 Model 3 是一部纯电动轿车，续航里程较长。
[3] iPhone 15 Pro 采用了钛金属边框，搭载 A17 Pro 芯片。
[4] 华为 Mate 60 Pro 支持卫星通话功能，采用麒麟芯片。
